# Explicit docking torsion policy — 2026-10-06

DockingMT #17/#6 adopts public MolSysMT #224 classification before H projection. Selected cuts remain caller decisions; provider exclusions are preserved even when an explicit docking exception is accepted. This notebook tests software contracts, not rotational barriers or predictive accuracy. Earlier fragment/type receipts remain intact.

In [1]:
from pathlib import Path
import sys
assert Path(sys.prefix).name == 'molsyssuite@uibcdf_3.14'
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(root))
import json
import pandas as pd
import dockingmt as dmt
from devtools.qualify_torsion_policy import qualify, save
record = qualify()
save(record)
print(record['interpreter'])
print(record['qualified_provider_commit'])

[17:19:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


/home/liliana/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python
5bd893c85fe8d211663b2b1f865f5f1d2c382a90


## Original reference matrix

The unchanged four curated SDF files use the pre-existing explicit RDKit reader bridge. Native classification itself uses no RDKit or Meeko. Every generated PDBQT hash and written permutation matches the pre-migration baseline; previous coordinate/element reference tests independently compare exact bond/fragment identities. The published files are comparison inputs, not chemical ground truth.

In [2]:
rows=[]
for name, case in record['original_cases'].items():
    selected=case['prepared']['metadata']['torsion_selection']['selected_bonds']
    rows.append({'case': name, 'provider_candidates': case['provider_candidate_count'], 'selected_cuts': len(selected), 'explicit_exceptions': sum(d['decision']=='explicit_override' for d in selected), 'prior_bytes_and_permutation': case['original_bytes_and_permutation_preserved'], 'parser': case['vina_parser_admitted']})
pd.DataFrame(rows)

,case,provider_candidates,selected_cuts,explicit_exceptions,prior_bytes_and_permutation,parser
0,1iep_ligand,7,7,0,True,True
1,1s63_ligand,5,6,1,True,True
2,5x72_ligand_p59,2,2,0,True,True
3,5x72_ligand_p69,2,2,0,True,True


## Explicit exceptions retain provider disagreement

The named consumer policy explicit_docking_cuts@1 retains explicitly requested ester/thioester C–O/S and triple-adjacent axes. The original provider mask and is_rotatable=False remain unchanged. These decisions do not silently redefine provider eligibility or activate any default cuts. Restricted C–N, ring, nonsingle, H and terminal-heavy bonds are rejected.

In [3]:
decisions=[]
for name, case in record['analytical_cases'].items():
    selection=case['prepared']['metadata']['torsion_selection']
    decision=selection['selected_bonds'][0]
    decisions.append({'case': name, 'bond_id': decision['source_bond_id'], 'source_pair': decision['source_atom_indices'], 'decision': decision['decision'], 'provider_exclusions': decision['provider_exclusion_reasons']})
pd.DataFrame(decisions)

,case,bond_id,source_pair,decision,provider_exclusions
0,"CCCCCC:(1, 2)",1,"[1, 2]",provider_candidate,[]
1,"CC(=O)OCC:(1, 3)",2,"[1, 3]",explicit_override,[restricted_conjugation]
2,"CC(=O)OCC:(3, 4)",3,"[3, 4]",provider_candidate,[]
3,"CC(=O)SCC:(1, 3)",2,"[1, 3]",explicit_override,[restricted_conjugation]
4,"c1ccccc1C#N:(5, 6)",5,"[5, 6]",explicit_override,[adjacent_triple_bond]
5,"CCC#CCC:(1, 2)",1,"[1, 2]",explicit_override,[adjacent_triple_bond]


## Restricted C–N scope

Amide/thioamide, amidine and tertiary-amide controls are rejected using the provider restricted_conjugation bit and the consumer C–N rule. Amidine rejection broadens the former local amide-only guard and is recorded as a behavior change. No symmetry exception or barrier calculation is introduced.

In [4]:
assert len(record['restricted_c_n_rejections']) == 4
record['restricted_c_n_rejections']

{'CC(=O)NCC': {'exception': 'ArgumentError',
  'message': "Invalid argument 'active_torsion_bonds': (1, 3) is an amide C-N or other restricted C-N bond."},
 'CC(=S)NCC': {'exception': 'ArgumentError',
  'message': "Invalid argument 'active_torsion_bonds': (1, 3) is an amide C-N or other restricted C-N bond."},
 'CC(=N)NCC': {'exception': 'ArgumentError',
  'message': "Invalid argument 'active_torsion_bonds': (1, 3) is an amide C-N or other restricted C-N bond."},
 'CC(=O)N(C)CC': {'exception': 'ArgumentError',
  'message': "Invalid argument 'active_torsion_bonds': (1, 3) is an amide C-N or other restricted C-N bond."}}

## Prepared and written atom axes

The complete selected source graph is evaluated before nonpolar-H projection. Source bond positions and IDs remain separate. ROOT/BRANCH orientation belongs to DockingMT; every written atom maps to the recorded source axis. The 1S63 reference-only H remains excluded from source-index comparison.

In [5]:
case=record['original_cases']['5x72_ligand_p59']
selection=case['prepared']['metadata']['torsion_selection']
order=case['prepared']['pdbqt_atom_indices']
assert selection['pdbqt_to_source_atom_indices'] == [selection['retained_source_atom_indices'][i] for i in order]
pd.DataFrame({'written_index': range(len(order)), 'prepared_index': order, 'selected_source_index': selection['pdbqt_to_source_atom_indices']})

,written_index,prepared_index,selected_source_index
0,0,7,7
1,1,8,8
2,2,9,9
3,3,10,10
4,4,11,11
5,5,12,12
6,6,13,13
7,7,14,14
8,8,15,15
9,9,16,16


## Automatic preparation with the default Vina gate

The preassigned named/charged 5X72 ligand reaches automatic preparation with two protocol-selected cuts under a pm/fs context. The unrelated external 1IEP receptor is a parser/execution fixture; this combination is not a biological redocking case. One pose, captured inputs and saved-result correspondence establish software interoperability only. Scientific preparation assessment remains unassessed.

In [6]:
saved=record['default_vina_result']
restored=dmt.DockingResult.from_dict(json.loads(json.dumps(saved,allow_nan=False)))
assert len(restored.poses)==1
assert dmt.verify_captured_inputs(restored.provenance['backend_artifacts'])==record['verified_captured_bytes']
selection=restored.provenance['preparation']['partner']['metadata']['torsion_selection']
assert selection['pdbqt_to_source_atom_indices']==restored.poses[0].metadata['selected_atom_indices']
print('poses:',len(restored.poses))
print('assessment:',restored.provenance['preparation']['partner']['assessment'])
print('captured input bytes:',record['verified_captured_bytes'])
selection['selected_bonds']

poses: 1
assessment: unassessed
captured input bytes: {'receptor': 216160, 'partner': 2740}


[{'source_atom_indices': [6, 7],
  'source_bond_index': 11,
  'source_bond_id': '11',
  'source_atom_ids': ['7', '8'],
  'decision': 'provider_candidate',
  'provider_exclusion_reasons': []},
 {'source_atom_indices': [17, 18],
  'source_bond_index': 30,
  'source_bond_id': '30',
  'source_atom_ids': ['18', '19'],
  'decision': 'provider_candidate',
  'provider_exclusion_reasons': []}]

## Provider-owned retained-connectivity gap

A declared assigned state splits a six-carbon chain into two components. Existing covalent-block defaults refer to the reference chain and cannot receive chemical_state/structure_indices. MolSysMT #348 owns the public state-aware induced-connectivity proposal. The existing local retained-axis traversal remains a tracked exception until that contract passes subset/assigned-state controls; review date 2027-01-06.

In [7]:
gap=record['state_scope_gap']
assert gap['assigned_fragment_offsets']==[0,3,6]
assert gap['reference_default_blocks']==[list(range(6))]
gap

{'owning_issue': 'uibcdf/molsysmt#348',
 'assigned_fragment_offsets': [0, 3, 6],
 'reference_default_blocks': [[0, 1, 2, 3, 4, 5]],
 'explicit_state_request': {'exception': 'UnknownArgumentError',
  'message': "'molsysmt.topology.get_covalent_blocks.get_covalent_blocks' does not accept the argument 'chemical_state'. Check the arguments it accepts. It accepts: molecular_system, output_type, remove_bonds, selection, syntax. See https://www.uibcdf.org/argdigest."},
 'source_unchanged': True}

## Source identity and limits

Loaded provider source bytes are checked against the committed 5bd893c85 source. Runtime distribution metadata and native build-provenance limitations remain separate. A bounded fetch found MolSysMT 14 commits behind remote, with consumed topology files unchanged; its worktree was preserved. No hosted, clean installed artifact or public support claim is made.

In [8]:
assert json.loads(json.dumps(record,allow_nan=False))==record
print(record['producer_versions'])
print(record['prior_source_profile'])
for limit in record['limits']:
    print('-',limit)

{'molsysmt': '0.22.4+215.g5bd893c85', 'argdigest': '0.15.0+1.g5c6711e', 'molsysviewer': '0.23.4+90.gc046fca1', 'rdkit': '2026.3.1', 'networkx': '3.7', 'vina': '1.2.7', 'pyunitwizard': '0.28.1+3.g2ab37a5'}
{'path': 'devguide/validation/data/chemical_templates/source_profile_2026-10-06.json', 'sha256': 'a201cf4b1429087cd714786b917a9f590b35bc8810ca5ee3e506971c8008236d'}
- Experimental provider conjugation_restricted@1 with explicit consumer overrides; no automatic torsion selection or barrier prediction.
- Original four SDFs retain the pre-existing explicit RDKit reader bridge; native graph classification itself imports no RDKit/Meeko.
- Original PDBQT byte parity is provisional software evidence, not chemically validated ground truth.
- The default Vina control pairs a named/charged 5X72 ligand with an unrelated external 1IEP receptor; it tests software/provenance, not biological redocking or affinity.
- Provider eligibility/metadata do not qualify scientific preparation/scoring; the def